# Sprint 54 Task 2 — Judge Score Precision

Kaggle notebook conversion of `Sprint54_Task2_Judge_Score_Precision.py`. It uses the shared `sprint54_judge_common.py` utility unchanged and the Sprint 51 `gold_set_labeled.csv` dataset.

**Before running:** enable a GPU accelerator (T4 x2 is preferred if available), enable Internet so Hugging Face can download Qwen3-8B, and attach a Kaggle dataset containing `gold_set_labeled.csv` and `sprint54_judge_common.py` (or upload these files to `/kaggle/working`). Run the setup cell, review the configuration, then run the experiment cell. The model load can take several minutes.

In [ ]:
from pathlib import Path
import sys, glob, os

# Locate input files whether attached as a Kaggle Dataset or uploaded to /kaggle/working.
def find_file(filename):
    candidates = [Path('/kaggle/working') / filename]
    candidates += [Path(p) for p in glob.glob('/kaggle/input/**/' + filename, recursive=True)]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(f'Could not find {filename}. Attach it as a Kaggle Dataset or upload it to /kaggle/working.')

DATA_PATH = str(find_file('gold_set_labeled.csv'))
COMMON_PATH = find_file('sprint54_judge_common.py')
sys.path.insert(0, str(COMMON_PATH.parent))
OUTPUT_DIR = '/kaggle/working/Sprint54_Task2_Score_Precision_Kaggle.ipynb'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Configuration: edit these before running the experiment.
DIRECTION = 'EN_CMN'
MAX_EXAMPLES = 50
DRY_RUN = False
DRY_RUN_N = 3
print('Dataset:', DATA_PATH)
print('Shared utility:', COMMON_PATH)
print('Output:', OUTPUT_DIR)
print('GPU available:', __import__('torch').cuda.is_available())


## Experiment configuration

For the first run, set `DRY_RUN = True` and `DRY_RUN_N = 3` to validate the pipeline. This still loads the Qwen model. After reviewing outputs, set `DRY_RUN = False` for the full run. Results are saved under `/kaggle/working/` and can be downloaded from the notebook output panel.

In [ ]:
# Provide command-line arguments expected by the original experiment logic.
import sys
argv = ['Sprint54_Task2_Judge_Score_Precision.py', '--data', DATA_PATH, '--output', OUTPUT_DIR, '--direction', DIRECTION, '--max-examples', str(MAX_EXAMPLES)]
if DRY_RUN:
    argv += ['--dry-run', '--dry-run-n', str(DRY_RUN_N)]
argv += ['--runs', '3', '--actionable-difference', '0.5']
sys.argv = argv
print('Arguments:', ' '.join(argv))


In [ ]:
"""Sprint 54 Task 2: measure repeatability and score precision of the committed P1 judge.

Example:
  python Sprint54_Task2_Judge_Score_Precision.py --data gold_set_labeled.csv --output sprint54_task2_outputs
"""
import argparse, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import t
from sprint54_judge_common import P1Judge, load_gold, environment_metadata, MODEL_NAME, MAX_SEQ_LENGTH, MAX_NEW_TOKENS, LOAD_IN_4BIT, PROMPT_ID

def main():
    ap=argparse.ArgumentParser(); ap.add_argument('--data',required=True); ap.add_argument('--output',default='sprint54_task2_outputs')
    ap.add_argument('--direction',default='EN_CMN'); ap.add_argument('--runs',type=int,default=3); ap.add_argument('--max-examples',type=int,default=50)
    ap.add_argument('--actionable-difference',type=float,default=0.5,help='Predeclared score difference to size the evaluation for (default 0.5/10).')
    ap.add_argument('--dry-run',action='store_true'); ap.add_argument('--dry-run-n',type=int,default=3)
    args=ap.parse_args()
    if args.runs<2: raise ValueError('--runs must be at least 2 to estimate within-example variation')
    if args.actionable_difference<=0: raise ValueError('--actionable-difference must be positive')
    out=Path(args.output); out.mkdir(parents=True,exist_ok=True)
    config={"task":"Sprint54_Task2","model_name":MODEL_NAME,"prompt_id":PROMPT_ID,"temperatures":[0.0,0.2],"runs_per_temperature":args.runs,"max_seq_length":MAX_SEQ_LENGTH,"max_new_tokens":MAX_NEW_TOKENS,"load_in_4bit":LOAD_IN_4BIT,"direction":args.direction,"actionable_difference_points":args.actionable_difference,"sample_size_planning_alpha":0.05,"sample_size_planning_power":0.80}
    df=load_gold(args.data,args.direction).sort_values('sample_id').head(args.max_examples).copy()
    if args.dry_run: df=df.head(args.dry_run_n)
    if df.empty: raise ValueError(f"No examples found for direction {args.direction}")
    metadata=environment_metadata(config,args.data); metadata['gold_rows_for_direction']=len(df)
    (out/'experiment_config.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    judge=P1Judge(); rows=[]
    for i,row in df.iterrows():
        for temp in (0.0,0.2):
            for run in range(args.runs):
                seed=54000 + int(i)*100 + run if temp==0 else 64000 + int(i)*100 + run
                result=judge.evaluate(row.source_en,row.mt_output,temperature=temp,seed=seed)
                rows.append({"sample_id":row.sample_id,"direction":row.get('direction',args.direction),"source_en":row.source_en,"candidate":row.mt_output,"human_overall_gold":row.overall_gold,
                  "temperature":temp,"run_id":run+1,"seed":seed,"judge_score":result['score'],"parse_ok":result['parse_ok'],"error":result.get('error'),"feedback":result.get('feedback'),"raw_output":result.get('raw_output')})
                print(f"{row.sample_id} temp={temp} run={run+1}/{args.runs}: {result['score']}")
    raw=pd.DataFrame(rows); raw.to_csv(out/'raw_runs.csv',index=False)
    valid=raw[raw.parse_ok & raw.judge_score.notna()].copy()
    per=[]
    for (sid,temp),g in raw.groupby(['sample_id','temperature'],sort=True):
        scores=g.loc[g.parse_ok & g.judge_score.notna(),'judge_score'].astype(float).to_numpy(); n=len(scores)
        per.append({"sample_id":sid,"temperature":temp,"runs_requested":args.runs,"valid_runs":n,"failed_runs":args.runs-n,
          "score_min":float(np.min(scores)) if n else None,"score_max":float(np.max(scores)) if n else None,"score_range":float(np.ptp(scores)) if n else None,
          "mean_score":float(np.mean(scores)) if n else None,"sample_sd":float(np.std(scores,ddof=1)) if n>=2 else None,
          "sem_within_example":float(np.std(scores,ddof=1)/math.sqrt(n)) if n>=2 else None,"scores_identical":bool(np.ptp(scores)==0) if n else None})
    per_df=pd.DataFrame(per); per_df.to_csv(out/'per_example_precision.csv',index=False)
    summaries=[]
    for temp,g in per_df.groupby('temperature'):
        complete=g[g.valid_runs>=2]
        all_scores=valid[valid.temperature.eq(temp)].groupby('sample_id').judge_score.mean()
        # Across-example SEM estimates precision of the set mean; it is distinct from within-example run variability.
        between_sd=float(all_scores.std(ddof=1)) if len(all_scores)>=2 else None
        set_sem=between_sd/math.sqrt(len(all_scores)) if between_sd is not None else None
        ranges=complete.score_range.dropna()
        sds=complete.sample_sd.dropna()
        summaries.append({"temperature":temp,"examples_attempted":int(g.sample_id.nunique()),"examples_with_2plus_valid_runs":int(len(complete)),
          "mean_score_range":float(ranges.mean()) if len(ranges) else None,"median_score_range":float(ranges.median()) if len(ranges) else None,
          "max_score_range":float(ranges.max()) if len(ranges) else None,"mean_within_example_sd":float(sds.mean()) if len(sds) else None,
          "examples_with_identical_runs":int(g.scores_identical.fillna(False).sum()),"valid_example_means":int(len(all_scores)),"between_example_sd_of_means":between_sd,"set_mean_sem":set_sem,
          "approx_95pct_margin_for_set_mean":1.96*set_sem if set_sem is not None else None})
    summary=pd.DataFrame(summaries); summary.to_csv(out/'precision_summary.csv',index=False)
    # Sample-size planning: one-sample approximation for detecting a set mean shift delta, based on observed between-example SD.
    # This is explicitly an estimate; use paired-difference SD when a future paired candidate-vs-baseline experiment is available.
    plans=[]
    for _,r in summary.iterrows():
        sd=r['between_example_sd_of_means']; delta=args.actionable_difference
        if pd.notna(sd) and sd>0:
            n_est=math.ceil(((1.96+0.8416)*sd/delta)**2)
            n_est=max(2,n_est)
        else: n_est=None
        plans.append({"temperature":r['temperature'],"actionable_difference_points":delta,"observed_between_example_sd":sd,"estimated_examples_for_80pct_power_two_sided_alpha_0.05":n_est,
          "current_examples":int(r['valid_example_means']),"current_set_sufficient_by_estimate":bool(n_est is not None and int(r['valid_example_means'])>=n_est) if n_est is not None else None,
          "method":"normal approximation: n=((1.96+0.8416)*SD/delta)^2; planning estimate, not a validated power analysis"})
    plan=pd.DataFrame(plans); plan.to_csv(out/'sample_size_analysis.csv',index=False)
    # Single actionable threshold: larger of the predeclared practical floor and twice the worst observed set-mean SEM.
    valid_sem=summary.set_mean_sem.dropna()
    observed_uncertainty=float(2*valid_sem.max()) if len(valid_sem) else 0.0
    actionable=max(args.actionable_difference,observed_uncertainty)
    lines=["# Sprint 54 Task 2 — Judge Score Precision Finding","",f"Judge: {MODEL_NAME}; prompt: {PROMPT_ID}; runs per example/temperature: {args.runs}.",f"Examples: {len(df)} EN→Mandarin gold-set rows. Temperatures: 0.0 and 0.2.","",f"**Minimum actionable score difference: {actionable:.3f}/10.** This is the larger of the predeclared practical floor ({args.actionable_difference:.3f}) and twice the largest observed set-mean SEM. It is a provisional operational threshold, not proof that smaller differences are meaningless.","","## Results","","| Temperature | Examples | Mean range | Max range | Mean within-example SD | Identical-run examples | Set-mean SEM | Approx. 95% margin |","|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for _,r in summary.iterrows():
        def fmt(v): return f"{v:.3f}" if pd.notna(v) else 'N/A'
        lines.append(f"| {r.temperature:.1f} | {int(r.valid_example_means)} | {fmt(r.mean_score_range)} | {fmt(r.max_score_range)} | {fmt(r.mean_within_example_sd)} | {int(r.examples_with_identical_runs)} | {fmt(r.set_mean_sem)} | {fmt(r.approx_95pct_margin_for_set_mean)} |")
    lines += ["","## Interpretation","","Within-example score range and SD measure run-to-run repeatability. The set-mean SEM instead describes uncertainty in the average score across this finite set of examples; these quantities must not be confused. If all temperature-0 scores repeat identically, the observed within-example SD is zero, but systematic judge errors remain possible. Temperature 0.2 can reveal sampling variability, though three runs give only a rough estimate.","","The sample-size table uses a normal-approximation planning calculation with the observed between-example SD and a predeclared difference. It is a planning proxy for the set mean, not a direct estimate of the number of examples needed for every paired model comparison. For a future baseline-versus-candidate decision, estimate the SD of paired per-example score differences and calculate power from those differences.","","See `experiment_config.json` for the configuration fingerprint, dataset hash, and environment versions; `raw_runs.csv` retains each raw judge response for audit.",""]
    (out/'findings.md').write_text('\n'.join(lines),encoding='utf-8')
    print('\nSaved outputs to',out.resolve())
main()


## Output files

Review `findings.md` first, then the CSV files for detailed results. Task 1: inspect `fault_cases.csv`; skipped/unscorable mutations are not passes. Task 2: compare within-example score ranges at temperature 0.0 vs 0.2, and treat sample-size calculations as planning estimates rather than a definitive power analysis.